# Bài 23 · Quy luật mở rộng: tự khớp luật của Chinchilla trên 75 GPT tí hon, rồi hỏi “20 token mỗi tham số” từ đâu ra

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/23-quy-luat-mo-rong.ipynb)

Notebook đi kèm [Bài 23 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/quy-luat-mo-rong/) (demo tương tác nằm ở trang bài học).
Mất mát của mô hình ngôn ngữ giảm đều đặn, đoán trước được, khi tăng số tham số $N$ và số token huấn luyện $D$,
nhưng không giảm mãi: nó có đáy.
Ta tự kiểm chứng điều đó trên một quét nhỏ có thật: các GPT theo ký tự học Wikipedia tiếng Việt. Ta khớp
$L(N, D) = E + A/N^\alpha + B/D^\beta$ đúng như Chinchilla làm, rồi dùng chính công thức ấy với các hằng số đã công bố.

**Bạn sẽ làm:**
1. Đếm tham số và FLOP: vì sao huấn luyện tốn $C \approx 6ND$, kiểm chứng trên GPT-2 small và Llama 3 8B.
2. Tự viết hàm mất mát Huber trong không gian log, gradient của nó và bộ tối ưu BFGS, rồi khớp lại đường của demo.
3. Thấy cái đáy $E$, và dùng đường khớp để đoán một mô hình lớn hơn chưa từng thấy.
4. Giải đường tối ưu tính toán (dạng đóng), so với thung lũng IsoFLOP, và tính lại “20 token mỗi tham số”
   từ ba bộ hằng số của Chinchilla.
5. Tính khi nào mô hình nhỏ học lâu (Llama 3 8B) lại rẻ hơn, và vì sao năng lực “bộc phát” phần lớn là do thước đo.

**Cần biết trước:** [Bài 04 · Softmax & hàm mất mát](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-mat-mat/)
(cross-entropy, nat), [Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/)
(các ma trận trong một khối) và
[Bài 22 · Tiền huấn luyện & tinh chỉnh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/tien-huan-luyen/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải ba tệp kết quả (~70 KB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra (hoặc đọc từ tệp kết quả mà demo dùng), và trùng với
số trong demo. Riêng 306 lần huấn luyện (khoảng 10 giờ CPU) được nạp từ đúng các tệp mà demo dùng; notebook khớp
lại, ngoại suy và tính mọi thứ còn lại ngay tại chỗ.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.
Hàm `sci()` in số rất lớn hoặc rất nhỏ kiểu $5{,}76 \cdot 10^{23}$.

In [ ]:
import hashlib
import itertools
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(v, digits=2):
    """5,76·10²³ · 10¹³ · 2,5·10⁻⁵"""
    e = math.floor(math.log10(abs(v)))
    m = round(v / 10.0**e, digits)
    if abs(m) >= 10:
        m, e = m / 10, e + 1
    head = "" if abs(m - 1) < 1e-12 else f"{vn(m, digits).rstrip('0').rstrip(',')}·"
    return f"{head}10{str(e).translate(SUP)}"


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(vn(13952, 0), vn(1.8344, 2), pct(0.0714, 0), sci(5.76e23), sci(1e13))

## 2 · Dữ liệu: 306 lần huấn luyện có thật

Demo của bài đã huấn luyện các GPT theo **ký tự** (mỗi token là một ký tự, Bài 14) trên 2.185 bài Wikipedia tiếng Việt
dài ít nhất 5.000 ký tự (Wikimedia, CC BY-SA). Kiến trúc là GPT của Bài 17 thu nhỏ: chuẩn hóa trước (pre-LN), GELU,
vị trí học được, ngữ cảnh 64 ký tự; AdamW với warmup rồi lịch cosin giảm $\eta$ còn 0,1 lần đúng sau $D$ ký tự,
như Chinchilla làm. Mỗi ô của lưới (cỡ mô hình × số ký tự) có $\eta$ riêng, chọn trên tập kiểm định; mọi mất mát
dưới đây đo trên tập thử riêng, các bài chưa từng học.

Huấn luyện lại hết mất cỡ 10 giờ CPU, nên ta nạp kết quả của demo (`fetch_json`, có kiểm tra MD5):
- `scaling-runs.json`: mất mát của mọi lần chạy, cả số đo “bộc phát” và văn bản mẫu;
- `scaling-fit.json`: các đường khớp, bootstrap và parabola IsoFLOP mà demo đã tính (để so với kết quả của ta);
- `scaling-chinchilla.json`: các hằng số đã công bố của Chinchilla và N, D của các mô hình thật.

In [ ]:
# Kết quả của một lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


RUNS = fetch_json("scaling-runs.json", "897d2bada61f2552e89d15f2631c422a")
FITS = fetch_json("scaling-fit.json", "7a5e470607ddc373654fe32a92385bcd")
CHIN = fetch_json("scaling-chinchilla.json", "00f11d8a052510080b7aebc94959d89c")
corpus, setup = RUNS["corpus"], RUNS["setup"]
print(
    f"{vn(corpus['kept'], 0)} bài; ký tự huấn luyện / kiểm định / thử: "
    + " / ".join(vn(corpus["chars"][k], 0) for k in ("train", "val", "test"))
    + f"; bảng chữ {corpus['vocab']} ký tự"
)
print(
    f"cả quét: {setup['nRuns']} lần huấn luyện, {vn(setup['cpuHours'], 1)} giờ CPU; batch {setup['B']} × {setup['T']} ký tự"
)
print(f"chỉ giữ bài dài ≥ {vn(corpus['minChars'], 0)} ký tự; cuối lịch cosin, η còn {vn(0.1, 1)} lần đỉnh")

## 3 · Ba núm vặn: $N$, $D$ và $C \approx 6ND$

- $N$: số tham số của mô hình.
- $D$: số token huấn luyện, ở đây là số **ký tự**.
- $C$: tổng số phép tính (FLOP) để huấn luyện. FLOP là *số* phép tính, còn FLOP/s là *tốc độ*.

Vì sao $C \approx 6ND$? Xét một lớp tuyến tính $Y = XW$ với $W$ cỡ $d_\text{in} \times d_\text{out}$, áp dụng cho $n$
token. Lượt xuôi tốn $n \cdot d_\text{in} d_\text{out}$ phép nhân–cộng (MAC), tức đúng một MAC cho mỗi trọng số và mỗi
token. Lượt ngược (Bài 06) cần **hai** phép nhân ma trận cùng cỡ: $\partial L/\partial X = (\partial L/\partial Y)\,W^\top$
để truyền tiếp về lớp trước, và $\partial L/\partial W = X^\top (\partial L/\partial Y)$ để cập nhật trọng số.
Cộng lại là 3 MAC = 6 FLOP cho mỗi trọng số và mỗi token. Ô dưới đếm MAC của ba phép nhân từ kích thước của chúng,
và kiểm tra gradient theo $W$ bằng sai phân hữu hạn.

In [ ]:
rng = np.random.default_rng(0)
n, d_in, d_out = 64, 48, 32
X, W = rng.normal(size=(n, d_in)), rng.normal(size=(d_in, d_out))
G = rng.normal(size=(n, d_out))  # ∂L/∂Y của một hàm mất mát L = Σ Y ⊙ G


def macs(A, B):
    """Phép nhân ma trận (p×q)·(q×r) tốn p·q·r phép nhân–cộng."""
    return A.shape[0] * A.shape[1] * B.shape[1]


Y = X @ W  # lượt xuôi
dX = G @ W.T  # lượt ngược: gradient theo đầu vào, truyền về lớp trước
dW = X.T @ G  # lượt ngược: gradient theo trọng số
counts = [macs(X, W), macs(G, W.T), macs(X.T, G)]
print("MAC xuôi / ngược theo X / ngược theo W:", " / ".join(vn(c, 0) for c in counts))
print(f"= {sum(counts) // (n * W.size)} MAC, tức {2 * sum(counts) // (n * W.size)} FLOP, cho mỗi trọng số và mỗi token")

E_ij = np.zeros_like(W)
E_ij[3, 5] = 1e-6
num = (((X @ (W + E_ij)) * G).sum() - ((X @ (W - E_ij)) * G).sum()) / 2e-6
print(f"∂L/∂W[3, 5]: công thức {vn(dW[3, 5], 6)}, sai phân hữu hạn {vn(num, 6)}")
assert abs(num - dW[3, 5]) < 1e-6 and sum(counts) == 3 * n * W.size

Cả mô hình là nhiều phép nhân ma trận như vậy, nên $C \approx 6 \cdot N \cdot D$. Công thức bỏ qua hai thứ:
- phần **attention** ($q \cdot k$ và tổng có trọng số của $v$, Bài 16), tỉ lệ với độ dài ngữ cảnh chứ không với $N$;
- việc **tra bảng embedding**: hàng $x_t$ của bảng được *chọn*, không được nhân, nên tham số của bảng gần như không tốn
  phép tính nào.

Ô dưới đếm tham số từ cấu hình đã công bố của GPT-2 small (Bài 17) và Llama 3 8B, rồi đếm đủ mọi phép nhân ma trận
cho một token huấn luyện. Attention lấy trung bình trên các vị trí $1 \dots$ ngữ cảnh, vì token thứ $t$ nhìn $t$ token.
Đây cũng là cách demo đếm.

In [ ]:
def params(c):
    """Số tham số theo phần. GPT-2: GELU, LayerNorm (tỉ lệ + dịch), bias, vị trí học được.
    Llama: SwiGLU (3 ma trận), RMSNorm (chỉ tỉ lệ), RoPE (không có tham số vị trí), không bias."""
    L, d, q, kv, ff, V = c["L"], c["d"], c["heads"] * c["hd"], c["kv"] * c["hd"], c["ff"], c["V"]
    gpt2 = c["mlp"] == "GELU"
    attn = L * (d * (q + 2 * kv) + q * d + ((q + 2 * kv + d) if gpt2 else 0))
    mlp = L * ((2 * d * ff + ff + d) if gpt2 else 3 * d * ff)
    norm = (2 * L + 1) * (2 * d if gpt2 else d)
    embed, pos = V * d, (c["ctx"] * d if gpt2 else 0)
    out = 0 if c["tied"] else V * d  # GPT-2 dùng lại bảng embedding (Eᵀ) làm lớp ra
    total = embed + pos + attn + mlp + norm + out
    return {"total": total, "nonemb": total - embed - pos - out}


def train_flop(c, ctx):
    """FLOP cho mỗi token huấn luyện: 6 FLOP cho mỗi MAC của mọi phép nhân ma trận."""
    L, d, q, kv, ff = c["L"], c["d"], c["heads"] * c["hd"], c["kv"] * c["hd"], c["ff"]
    blocks = L * (d * (q + 2 * kv) + q * d + (2 if c["mlp"] == "GELU" else 3) * d * ff)
    out = d * c["V"]  # lớp ra: logit cho mọi token của từ điển
    attn = 2 * L * q * (ctx + 1) / 2  # q·kᵀ rồi tổng có trọng số của v, trên trung bình (ctx + 1)/2 vị trí
    return {"total": 6 * (blocks + out + attn), "attn": 6 * attn}


def config(L, d, heads, kv, hd, ff, V, ctx, mlp, tied):
    """Số khối, độ rộng, số đầu Q, số đầu K/V, cỡ mỗi đầu, độ rộng MLP, từ điển, ngữ cảnh, loại MLP, lớp ra = Eᵀ?"""
    return {"L": L, "d": d, "heads": heads, "kv": kv, "hd": hd, "ff": ff, "V": V, "ctx": ctx, "mlp": mlp, "tied": tied}


# OpenAI hparams.json (GPT-2 124M) và config.json của Llama 3 8B (Grattafiori và cộng sự 2024, bảng 3)
GPT2 = config(12, 768, 12, 12, 64, 3072, 50257, 1024, "GELU", tied=True)
LLAMA3_8B = config(32, 4096, 32, 8, 128, 14336, 128256, 8192, "SwiGLU", tied=False)
for name, c in (("GPT-2 small", GPT2), ("Llama 3 8B", LLAMA3_8B)):
    p, f = params(c), train_flop(c, c["ctx"])
    print(
        f"{name}: N = {vn(p['total'], 0)} (không embedding {vn(p['nonemb'], 0)}); ngữ cảnh {vn(c['ctx'], 0)}: "
        f"{vn(f['total'], 0)} FLOP/token = {vn(f['total'] / (6 * p['total']), 3)} × 6N, attention {pct(f['attn'] / f['total'], 1)}"
    )
gp, gf = params(GPT2), train_flop(GPT2, 1024)
print(f"GPT-2 small, 6N không tính embedding: {vn(6 * gp['nonemb'] / 1e6, 0)} triệu FLOP/token")
TABLE_A4 = [1.03, 1.10, 1.08, 1.04, 1.03, 0.99]  # Hoffmann và cộng sự 2022, bảng A4: FLOP đếm đủ / 6ND, 73M…6,8B
print(f"Chinchilla, bảng A4: đếm đủ / 6ND = {vn(min(TABLE_A4), 2)}–{vn(max(TABLE_A4), 2)}")
print(
    f"GPT-2 small, đếm đủ / 6N = {vn(gf['total'] / (6 * gp['total']), 2)}; bảng embedding của Llama 3 8B: "
    f"{vn(LLAMA3_8B['V'], 0)} × {vn(LLAMA3_8B['d'], 0)} = {vn(LLAMA3_8B['V'] * LLAMA3_8B['d'] / 1e6, 0)} triệu tham số"
)
assert (gp["total"], gp["nonemb"], gf["total"]) == (124439808, 85056000, 797870592), "khác với demo"
assert params(LLAMA3_8B)["total"] == 8030261248, "khác với demo"

GPT-2 small có đúng 124.439.808 tham số, như mô hình thật. Đếm đủ ra 797.870.592 FLOP cho mỗi token, gấp 1,07 lần
$6N$. Khoảng 7% là attention, và phần đó lớn dần theo ngữ cảnh (12,5% với ngữ cảnh 8.192 của Llama 3). Bảng A4 của Chinchilla cũng thấy tỉ số
“đếm đủ / $6ND$” nằm trong khoảng 0,99–1,10. Nếu không đếm bảng embedding như Kaplan, $6N$ chỉ còn 510 triệu, thấp hơn
thật khá xa, vì lớp ra của GPT-2 *chính là* bảng embedding chuyển vị, và lớp ra thì có nhân ma trận.

Bảng embedding của Llama 3 8B có 525 triệu tham số, thêm một lớp ra cùng cỡ: hơn 1 tỉ trên 8 tỉ.
Cách đếm $N$ (có hay không có embedding) sẽ quay lại ở mục 10.

## 4 · Tự đo: mất mát theo $D$ cho từng cỡ mô hình

Lưới của demo: 5 cỡ mô hình (độ rộng $d$ = 16…96) × 5 lượng dữ liệu ($D = 2^{18} \dots 2^{22}$ ký tự) × 3 seed.
Mỗi lần chạy có lịch learning rate dài **đúng bằng** $D$ của nó. Ngoài lưới còn một cỡ lớn hơn, $d = 128$, mà ta
không dùng để khớp: nó là bài kiểm tra cho việc ngoại suy ở mục 7.

In [ ]:
size = {s["d"]: s for s in RUNS["sizes"]}
grid = [r for r in RUNS["runs"] if r["kind"] == "grid"]
held = [r for r in RUNS["runs"] if r["kind"] == "held"]
iso_runs = [r for r in RUNS["runs"] if r["kind"] == "iso"]
DS = setup["grid"]
DMAX = DS[-1]
SIZES = sorted({r["d"] for r in grid})
for d, s in size.items():
    tag = "" if d in SIZES else "  ← giữ riêng, không dùng để khớp"
    print(f"d {d:>3}: {s['L']} khối, N = {vn(s['total'], 0):>7} (không embedding {vn(s['nonemb'], 0):>7}){tag}")
print(f"{len(grid)} lần chạy trong lưới, {len(held)} của cỡ giữ riêng, {len(iso_runs)} cho IsoFLOP (mục 9)")
print("D (ký tự):", ", ".join(vn(D, 0) for D in DS))


def cell(d, D, runs=grid + held):
    return np.array([r["test"] for r in runs if r["d"] == d and r["D"] == D])


assert all(len(cell(d, D)) == 3 for d in SIZES for D in DS)

Hình dưới vẽ mất mát thử (nat mỗi ký tự, trung bình 3 seed; thanh dọc là khoảng giữa 3 seed) theo $D$, mỗi cỡ một
màu và một hình điểm, trên thang log–log. Một luật lũy thừa thuần túy $L = B/D^\beta$ là một đường thẳng trên thang
này.

In [ ]:
MARKS = ["o", "s", "^", "D", "v", "P"]
RAMP = ["#a9c9f2", "#7aaced", "#3987e5", "#2563b0", "#173f73"]  # nhạt → đậm: cỡ nhỏ → lớn


def plot_grid(ax, minus=0.0):
    for i, d in enumerate(SIZES):
        m = np.array([cell(d, D).mean() for D in DS]) - minus
        lo = m - (np.array([cell(d, D).min() for D in DS]) - minus)
        hi = (np.array([cell(d, D).max() for D in DS]) - minus) - m
        ax.errorbar(DS, m, yerr=[lo, hi], color=RAMP[i], marker=MARKS[i], ms=6, lw=1.5, capsize=2,
                    label=f"d {d}: {vn(size[d]['total'], 0)} tham số")  # fmt: skip
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("D (ký tự huấn luyện, thang log)")


def pow2_ticks(ax):
    """Vạch 2¹⁸ … 2²² trên trục D; gọi sau vn_axes."""
    ax.set_xticks(DS, [f"2{str(round(math.log2(D))).translate(SUP)}" for D in DS])
    ax.xaxis.set_minor_formatter(NullFormatter())


fig, ax = plt.subplots(figsize=(8, 4.2))
plot_grid(ax)
ax.set(ylabel="L thử (nat/ký tự, thang log)", title="Mất mát thử theo dữ liệu, 5 cỡ mô hình × 3 seed")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.3, which="both")
vn_axes(fig)
ax.set_yticks([1.8, 2.0, 2.2, 2.4, 2.6])  # sau vn_axes: trục log ngắn này chỉ có một vạch 1–2–5
pow2_ticks(ax)
plt.show()
m16 = [cell(16, D).mean() for D in DS]
print("d 16, mỗi lần gấp đôi D bớt:", " · ".join(vn(a - b, 3) for a, b in itertools.pairwise(m16)), "nat")
print(f"d 16 học {vn(DMAX, 0)} ký tự: L = {vn(m16[-1], 3)}")
gaps = [np.ptp([cell(d, D).mean() for d in SIZES]) for D in (DS[0], DMAX)]
print(f"cỡ nhỏ nhất và lớn nhất cách nhau {vn(gaps[0], 2)} nat ở D = 2¹⁸ (128 bước), {vn(gaps[1], 2)} ở D = 2²²")
spread = [np.ptp(cell(d, D)) for d in SIZES for D in DS]
print(f"3 seed của một ô khác nhau trung vị {vn(np.median(spread), 3)}, nhiều nhất {vn(max(spread), 3)} nat")

Mọi đường đều đi xuống, cỡ lớn hơn nằm thấp hơn, nhưng đường của mô hình nhỏ nhất **cong dần**: lần gấp đôi dữ liệu
đầu bớt 0,159 nat, lần cuối chỉ bớt 0,065. Chênh lệch giữa 3 seed của một ô (trung vị 0,014 nat) nhỏ hơn nhiều so
với khoảng cách giữa các ô, nên hình dạng này là thật, không phải nhiễu. Ở $D$ nhỏ nhất, chỉ 128 bước huấn luyện, các
cỡ chỉ cách nhau 0,08 nat (ở $2^{22}$ là 0,26): chưa học đủ thì mô hình lớn chưa kịp có lợi.

## 5 · Khớp $L(N, D) = E + A/N^\alpha + B/D^\beta$ như Chinchilla

Hoffmann và cộng sự (2022) đề xuất dạng

$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$

- $E$ là đáy: mất mát còn lại khi cả $N$ và $D$ tiến ra vô cùng.
- $A/N^\alpha$ là phần do mô hình chưa đủ lớn, $B/D^\beta$ là phần do chưa học đủ dữ liệu.

Cách khớp (cách thứ 3 của bài báo, phụ lục D.2):
- Đặt $A = e^a$, $B = e^b$, $E = e^e$ để ba hằng số luôn dương. Khi đó
  $\ln L = \mathrm{LSE}(a - \alpha \ln N,\ b - \beta \ln D,\ e)$, với $\mathrm{LSE}(u) = \ln \sum_k e^{u_k}$.
  Tính LSE bằng cách trừ $\max u$ trước để không tràn số.
- So sánh **trong không gian log**: phần dư $r = \ln \hat L - \ln L$ là sai số tương đối.
- Cực tiểu hóa $\sum_i \mathrm{Huber}_\delta(r_i)$ với $\delta = 10^{-3}$: bậc hai khi $|r| \le \delta$, tuyến tính
  khi lớn hơn, nên một lần chạy lệch hẳn chỉ kéo đường khớp nhẹ thôi.

Ô dưới viết hàm mục tiêu cùng gradient của nó theo $\theta = (a, b, e, \alpha, \beta)$, rồi kiểm tra gradient bằng sai
phân hữu hạn.

In [ ]:
DELTA = 1e-3


def objective(th, lN, lD, lL, delta=DELTA):
    """Σ Huber_δ(LSE(a − α·ln N, b − β·ln D, e) − ln L) và gradient theo θ = (a, b, e, α, β)."""
    a, b, e, al, be = th
    u = np.stack([a - al * lN, b - be * lD, np.full_like(lN, e)])  # ln của ba số hạng A/N^α, B/D^β, E
    m = u.max(0)
    w = np.exp(u - m)
    s = w.sum(0)
    r = m + np.log(s) - lL  # LSE ổn định số: ln(dự đoán) − ln(đo)
    w /= s  # tỉ phần của mỗi số hạng trong dự đoán = ∂LSE/∂u
    inside = np.abs(r) <= delta
    f = np.where(inside, 0.5 * r * r, delta * (np.abs(r) - 0.5 * delta)).sum()
    h = np.where(inside, r, delta * np.sign(r))  # Huber′(r)
    g = np.array([(h * w[0]).sum(), (h * w[1]).sum(), (h * w[2]).sum(), -(h * w[0] * lN).sum(), -(h * w[1] * lD).sum()])
    return float(f), g


def points(runs, count="total", sched="sched"):
    """(N, D, L) cho mỗi lần chạy; N đếm mọi tham số ("total") hoặc không tính embedding ("nonemb")."""
    longest = {(r["d"], r["seed"]): r for r in grid if r["D"] == DMAX}
    out = []
    for r in runs:
        L = r["test"]
        if sched == "inter" and r["D"] < DMAX:  # kiểu Kaplan (mục 10): đọc giữa chừng lần chạy dài nhất
            L = next(t for ch, _, t in longest[(r["d"], r["seed"])]["inter"] if ch == r["D"])
        out.append((size[r["d"]][count], r["D"], L))
    return np.array(out, float)


P_all = points(grid)
lN, lD, lL = np.log(P_all.T)
th = np.array([5.0, 7.0, 0.4, 0.7, 0.5])
f, g = objective(th, lN, lD, lL)
num = []
for i in range(5):
    step = np.zeros(5)
    step[i] = 1e-6
    num.append((objective(th + step, lN, lD, lL)[0] - objective(th - step, lN, lD, lL)[0]) / 2e-6)
rel = np.abs(np.array(num) - g) / (np.abs(num) + np.abs(g))
print(
    f"{len(P_all)} điểm; mục tiêu tại θ thử = {vn(f, 5)}; sai lệch tương đối lớn nhất của gradient: {sci(rel.max(), 1)}"
)
assert rel.max() < 1e-5

Gradient khớp với sai phân hữu hạn. Giờ ta cần một bộ tối ưu. Năm tham số, hàm trơn (trừ chỗ nối của Huber): BFGS là
lựa chọn tự nhiên. Nó giữ một ước lượng $H$ của **nghịch đảo** ma trận Hessian, đi theo hướng $p = -Hg$, chọn bước
bằng tìm kiếm lùi (Armijo), rồi cập nhật $H$ từ độ dịch $s$ và độ đổi gradient $y$ của bước vừa đi. Đây cũng là bộ
tối ưu mà demo chạy trong trình duyệt.

Hàm mục tiêu có nhiều cực tiểu địa phương, nên ta xuất phát từ 32 điểm khác nhau và giữ kết quả tốt nhất. Chinchilla
dùng một lưới lớn hơn; 32 điểm này là những điểm demo dùng khi khớp lại trực tiếp.

In [ ]:
def bfgs(fn, x0, gtol=1e-13, iters=1000):
    """Cực tiểu fn (trả về giá trị và gradient) bằng BFGS với tìm kiếm lùi Armijo."""
    x = np.array(x0, float)
    f, g = fn(x)
    H = np.eye(len(x))
    for _ in range(iters):
        if np.abs(g).max() < gtol:
            break
        p = -H @ g
        if not p @ g < 0:  # H hết xác định dương: quay về hướng dốc nhất
            H, p = np.eye(len(x)), -g
        t = 1.0
        for _ in range(60):
            f_new, g_new = fn(x + t * p)
            if np.isfinite(f_new) and f_new <= f + 1e-4 * t * (p @ g):
                break
            t *= 0.5
        else:
            break  # không tìm được bước nào làm f giảm
        s, y = t * p, g_new - g
        x, f, g = x + s, f_new, g_new
        sy = s @ y
        if sy > 1e-300:
            Hy = H @ y
            H += (sy + y @ Hy) * np.outer(s, s) / sy**2 - (np.outer(Hy, s) + np.outer(s, Hy)) / sy
        if np.abs(s).max() < 1e-15:
            break
    return x, f


STARTS = list(itertools.product([0, 10], [5, 15], [-0.5, 0.5], [0.3, 1.0], [0.3, 1.0]))  # (a, b, e, α, β)


def fit(P, scale=1.0, gtol=1e-13, starts=STARTS, delta=DELTA):
    """Khớp E + A/N^α + B/D^β lên các điểm (N, D, L); scale = 1/len(P) là lấy trung bình thay vì tổng (mục 14)."""
    lN, lD, lL = np.log(P.T)

    def fn(th):
        f, g = objective(th, lN, lD, lL, delta)
        return scale * f, scale * g

    th, f = min((bfgs(fn, x0, gtol) for x0 in starts), key=lambda r: r[1])
    a, b, e, al, be = th
    return {"E": np.exp(e), "A": np.exp(a), "B": np.exp(b), "alpha": al, "beta": be, "obj": f / scale, "theta": th}


def predict(f, N, D):
    return f["E"] + f["A"] / np.asarray(N, float) ** f["alpha"] + f["B"] / np.asarray(D, float) ** f["beta"]


def show(f):
    return " · ".join(
        f"{k} = {vn(f[k], 4 if k in ('alpha', 'beta', 'E') else 1)}" for k in ("E", "A", "B", "alpha", "beta")
    )


f0 = fit(P_all)
stored = FITS["fits"]["all-total-sched"]
print("ta khớp:  ", show(f0).replace("alpha", "α").replace("beta", "β"))
print("demo lưu: ", show(stored).replace("alpha", "α").replace("beta", "β"))
res = predict(f0, P_all[:, 0], P_all[:, 1]) - P_all[:, 2]
inside = np.abs(np.log(predict(f0, P_all[:, 0], P_all[:, 1]) / P_all[:, 2])) <= DELTA
print(
    f"phần dư: trung bình bình phương {vn(np.sqrt(np.mean(res**2)), 3)} nat; {inside.sum()}/{len(res)} điểm trong |r| ≤ δ"
)
assert f0["obj"] <= stored["obj"] * (1 + 1e-5), "khác với demo"  # cùng cực tiểu với scipy của demo
assert all(abs(f0[k] - stored[k]) < 2e-3 for k in ("E", "alpha", "beta")), "khác với demo"

Ta khớp lại ra đúng đường của demo (sai khác ở chữ số thứ tư của $A$ và $B$, do hai bộ tối ưu dừng ở hai điểm rất
gần nhau). Năm hằng số tóm tắt cả 75 lần chạy, lệch trung bình (căn của trung bình bình phương) 0,035 nat. Chỉ khoảng 1/7 số điểm nằm trong vùng
bậc hai $|r| \le \delta$ của Huber, nên phần lớn điểm được tính theo $|r|$: với $\delta = 10^{-3}$ rất nhỏ, đây gần
như khớp theo trị tuyệt đối của sai số tương đối.

Ý nghĩa của $\alpha$: gấp đôi $N$ thì $A/N^\alpha$ nhân với $2^{-\alpha}$. Lưu ý đây chỉ là phần *còn giảm được*. Kaplan
và cộng sự (2020) khớp $L(N) = (N_c/N)^{\alpha_N}$ cho cả mất mát, không có đáy, và đo $\alpha_N \approx 0{,}076$:
gấp đôi $N$ thì cả $L$ nhân với $2^{-0{,}076} \approx 0{,}95$. Hai số mũ thuộc hai dạng công thức khác nhau, nên không so
trực tiếp được.

In [ ]:
print(f"2^(−α) = {vn(2 ** -f0['alpha'], 3)}: gấp đôi N, phần A/N^α còn lại {pct(2 ** -f0['alpha'], 0)}")
print(f"2^(−β) = {vn(2 ** -f0['beta'], 3)}: gấp đôi D, phần B/D^β còn lại {pct(2 ** -f0['beta'], 0)}")
print(f"Kaplan: 2^(−0,076) = {vn(2**-0.076, 3)}")

## 6 · Có đáy: trừ $E$ đi mới là đường thẳng

Trên thang log–log ở mục 4, các đường cong dần vì mất mát không thể xuống dưới
$E + A/N^\alpha$, đáy của một cỡ mô hình khi học vô hạn dữ liệu. Nếu dạng của Chinchilla đúng, thì trừ cả $E$ lẫn phần
của $N$ đi, mọi lần chạy phải nằm trên **cùng một đường thẳng** $B/D^\beta$, dốc $-\beta$, bất kể cỡ mô hình.

Hình dưới, bên trái: $L$ đo được và đáy $E + A/N^\alpha$ của từng cỡ (đường chấm ngang). Bên phải:
$L - E - A/N^\alpha$ của cùng các ô (trung bình 3 seed), mỗi cỡ một hình điểm, cùng đường $B/D^\beta$ (đen).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
plot_grid(a1)
for i, d in enumerate(SIZES):
    a1.axhline(f0["E"] + f0["A"] / size[d]["total"] ** f0["alpha"], color=RAMP[i], ls=":", lw=1.2)
a1.set(ylabel="L thử (nat/ký tự)", title="L và đáy E + A/N^α của từng cỡ")
a1.legend(frameon=False, fontsize=8)
for i, d in enumerate(SIZES):
    m = np.array([cell(d, D).mean() for D in DS])
    a2.plot(DS, m - f0["E"] - f0["A"] / size[d]["total"] ** f0["alpha"], MARKS[i], color=RAMP[i], ms=7, mec="white")
Dl = np.geomspace(DS[0] / 1.3, DS[-1] * 1.3, 50)
a2.plot(Dl, f0["B"] / Dl ** f0["beta"], color="black", lw=1.2, label=f"B/D^β, β = {vn(f0['beta'], 2)}")
a2.set(xscale="log", yscale="log", xlabel="D (ký tự, thang log)", ylabel="L − E − A/N^α (nat/ký tự)")
a2.set_title("Trừ đáy đi: mọi cỡ trên một đường thẳng")
a2.legend(frameon=False)
for ax in (a1, a2):
    ax.grid(alpha=0.3, which="both")
plt.tight_layout()
vn_axes(fig)
a1.set_yticks([1.6, 1.8, 2.0, 2.2, 2.4, 2.6])
pow2_ticks(a1)
pow2_ticks(a2)
plt.show()
floor16 = f0["E"] + f0["A"] / size[16]["total"] ** f0["alpha"]
print(f"E = {vn(f0['E'], 3)} nat/ký tự = {vn(f0['E'] / math.log(2), 2)} bit/ký tự")
print(
    f"đáy của d 16 (13.952 tham số): {vn(floor16, 3)}; nó đang ở {vn(m16[-1], 3)}, còn cách {vn(m16[-1] - floor16, 3)}"
)
assert abs(floor16 - 1.834) < 2e-3, "khác với demo"

Bên phải, 25 ô của 5 cỡ mô hình gần như nằm trên một đường thẳng, lệch nhiều nhất ở hai đầu (nhất là các lần chạy
ngắn nhất, 128 bước): phần do dữ liệu là một luật lũy thừa thật, giống nhau cho mọi cỡ. Bên trái, mô hình nhỏ nhất có đáy khoảng 1,83 nat/ký tự. Thêm dữ liệu bao nhiêu nó cũng không
xuống dưới được, vì giới hạn nằm ở sức chứa của 13.952 tham số.

Còn $E$ thì sao? Nó là đáy của **đường khớp này, trên dữ liệu này, với loại mô hình này**, không phải “entropy của
tiếng Việt”. Mục 7 sẽ thấy $E$ đổi tới 0,15 khi ta chỉ đổi tập lần chạy dùng để khớp, và mục 10 thấy nó xuống 1,44
khi chỉ đổi cách đọc mất mát.

## 7 · Ngoại suy: đoán một mô hình chưa từng thấy

Giá trị thật của một quy luật mở rộng là **dự đoán**: khớp trên các mô hình nhỏ, rẻ, rồi đoán mô hình lớn trước khi
huấn luyện nó. Mô hình giữ riêng ($d = 128$, 850.880 tham số) lớn gấp 1,7 lần cỡ lớn nhất đã khớp ($d = 96$),
và có 3 seed ở $D = 2^{21}$ và $2^{22}$ ký tự.

### 🤔 Dự đoán trước
Đường khớp đoán mất mát của mô hình giữ riêng ở $D = 2^{22}$. Nó sai khoảng bao nhiêu: dưới 0,05 nat, khoảng 0,2 nat,
hay hơn 0,5 nat?

In [ ]:
def signed(x, digits=3):
    return ("+" if x > 0 else "") + vn(x, digits)


NH = size[128]["total"]
print(f"mô hình giữ riêng: {vn(NH, 0)} tham số = {vn(NH / size[SIZES[-1]]['total'], 1)} lần cỡ lớn nhất đã khớp")
for D in (DS[-2], DMAX):
    meas, p = cell(128, D), predict(f0, NH, D)
    print(
        f"  D = {vn(D, 0)}: đoán {vn(p, 3)}, đo {vn(meas.min(), 3)}–{vn(meas.max(), 3)} qua 3 seed (trung bình {vn(meas.mean(), 3)}); "
        f"sai {signed(p - meas.mean())} nat = {pct(abs(p - meas.mean()) / meas.mean(), 1)} mất mát; "
        f"dao động giữa các seed {vn(np.ptp(meas), 3)}"
    )
err_held = predict(f0, NH, DMAX) - cell(128, DMAX).mean()
assert abs(err_held - 0.0174) < 2e-3, "khác với demo"

Đường khớp chưa từng thấy mô hình này mà vẫn đoán sai chưa tới 0,02 nat:
- Ở $2^{21}$ ký tự, sai +0,002, nằm trong dao động giữa các seed.
- Ở $2^{22}$ ký tự, sai +0,017. Con số này lớn hơn dao động giữa 3 seed (0,004), nhưng chỉ bằng 1% mất mát.

Đường khớp đoán hơi bi quan (mô hình thật tốt hơn dự đoán), và đó là hướng sai an toàn hơn khi phải quyết định có
huấn luyện mô hình lớn hay không.

Một lần đoán đúng chưa nói lên nhiều. Ô dưới khớp lại trên bốn tập con của lưới, mỗi lần giấu đi một phần, rồi đoán
phần bị giấu và mô hình giữ riêng.

In [ ]:
PRESETS = {
    "mọi lần chạy d ≤ 96": ("all", lambda r: True),
    "chỉ D ≤ 2,1 triệu ký tự": ("smallD", lambda r: r["D"] <= DS[-2]),
    "chỉ d ≤ 48": ("smallN", lambda r: r["d"] <= 48),
    "bỏ các lần chạy ≤ 256 bước": ("long", lambda r: r["steps"] > 256),
}
sub = {}
for name, (key, keep) in PRESETS.items():
    f = f0 if key == "all" else fit(points([r for r in grid if keep(r)]))
    sub[key] = f
    hidden = [r for r in grid if not keep(r)]
    line = f"{name:<27}: E {vn(f['E'], 3)} · α {vn(f['alpha'], 2)} · β {vn(f['beta'], 2)} → d 128 ở 2²²: {vn(predict(f, NH, DMAX), 3)}"
    if hidden:
        e = [predict(f, size[r["d"]]["total"], r["D"]) - r["test"] for r in hidden]
        w = int(np.argmax(np.abs(e)))
        line += f"; {len(hidden)} lần chạy bị giấu sai nhiều nhất {signed(e[w])} (d {hidden[w]['d']}, {vn(hidden[w]['steps'], 0)} bước)"
    print(line)
    st = FITS["fits"][f"{key}-total-sched"]
    assert abs(f["alpha"] - st["alpha"]) < 2e-3 and abs(f["beta"] - st["beta"]) < 2e-3, "khác với demo"
Es = [f["E"] for f in sub.values()]
heldp = [predict(f, NH, DMAX) for f in sub.values()]
print(
    f"E: {vn(min(Es), 2)}–{vn(max(Es), 2)} (chênh {vn(max(Es) - min(Es), 2)}); dự đoán cho d 128: chênh {vn(np.ptp(heldp), 3)} nat"
)

Bốn đường khớp cho các hằng số khác nhau xa: $E$ chênh 0,15, $\beta$ từ 0,47 đến 0,77. Vậy mà cả bốn đoán mô hình giữ
riêng trong một khoảng chỉ rộng 0,024 nat. **Dự đoán bền hơn nhiều so với số mũ.** Mấy chục lần chạy trên hai bậc
độ lớn đủ để đoán gần vùng đã đo, nhưng chưa đủ để xác định một “định luật” với số mũ chính xác.

Chỗ sai nhất (+0,30 nat) là lần chạy 128 bước của mô hình nhỏ nhất, khi ta chỉ khớp trên các lần chạy dài. Dạng
$E + A/N^\alpha + B/D^\beta$ không tả được những lần huấn luyện quá ngắn, khi warmup và lịch learning rate chiếm gần
hết thời gian. Besiroglu và cộng sự (2024) cũng loại năm lần chạy lạ của Chinchilla (dưới 0,4 token mỗi tham số, mất
mát cao bất thường) khỏi bản khớp lại của họ.

## 8 · Đường tối ưu tính toán: một lời giải dạng đóng

Giờ là câu hỏi Chinchilla đặt ra: với ngân sách cố định $C = 6ND$, nên chia cho $N$ và $D$ thế nào? Thay
$D = C/(6N)$ vào đường khớp, ta được một hàm một biến

$$L(N) = E + A N^{-\alpha} + B \left(\tfrac{C}{6}\right)^{-\beta} N^{\beta}.$$

Đặt đạo hàm bằng 0: $\alpha A N^{-\alpha-1} = \beta B (C/6)^{-\beta} N^{\beta-1}$, suy ra

$$N_\text{opt} = G \left(\tfrac{C}{6}\right)^{\beta/(\alpha+\beta)}, \qquad D_\text{opt} = G^{-1} \left(\tfrac{C}{6}\right)^{\alpha/(\alpha+\beta)}, \qquad G = \left(\tfrac{\alpha A}{\beta B}\right)^{1/(\alpha+\beta)}.$$

Ta kiểm tra lời giải bằng cách dò 400.001 giá trị $N$ dọc đường cùng ngân sách.

In [ ]:
def frontier(f, C, k=6.0):
    """N, D tối ưu tính toán và mất mát dự đoán ở ngân sách C = k·N·D."""
    al, be = f["alpha"], f["beta"]
    G = (al * f["A"] / (be * f["B"])) ** (1 / (al + be))
    N = G * (C / k) ** (be / (al + be))
    D = (C / k) ** (al / (al + be)) / G
    return N, D, float(predict(f, N, D))


def brute(f, C, N_guess, k=6.0):
    Ns = np.exp(np.linspace(np.log(N_guess) - 6, np.log(N_guess) + 6, 400001))
    L = predict(f, Ns, C / (k * Ns))
    return Ns[L.argmin()], L.min()


BUDGETS = sorted({r["C"] for r in iso_runs})
for C in BUDGETS:
    N, D, L = frontier(f0, C)
    Nb, Lb = brute(f0, C, N)
    print(
        f"C = {sci(C, 1)}: N_opt = {vn(N, 0)}, D_opt = {vn(D, 0)}, D/N = {vn(D / N, 0)}, L = {vn(L, 3)}  (dò: N = {vn(Nb, 0)})"
    )
    assert abs(math.log(Nb / N)) < 1e-3 and Lb >= L - 1e-12
print(f"(mỗi lần dò {vn(400001, 0)} giá trị N, cách đều theo ln N quanh lời giải)")
a_exp = f0["beta"] / (f0["alpha"] + f0["beta"])
print(f"a = β/(α+β) = {vn(a_exp, 3)}: N_opt ∝ C^{vn(a_exp, 2)}, D_opt ∝ C^{vn(1 - a_exp, 2)}")
print(
    "D/N tối ưu khi C = 10¹¹ … 10¹⁴:",
    " → ".join(vn(frontier(f0, C)[1] / frontier(f0, C)[0], 0) for C in 10.0 ** np.arange(11, 15)),
)

Lời giải dạng đóng trùng với phép dò. Hai điều cần để ý:
- Nếu $\alpha = \beta$ thì $a = 1/2$: $N$ và $D$ tăng cùng nhịp, mỗi khi $C$ gấp 4 thì cả hai gấp đôi, và tỉ lệ
  $D/N$ không đổi. Đó là kết luận chính của Chinchilla.
- Đường khớp của ta có $\alpha > \beta$, nên $a < 1/2$: tỉ lệ $D/N$ tối ưu **tăng dần** theo $C$, từ khoảng 120 lên
  khoảng 320 ký tự mỗi tham số khi $C$ tăng ba bậc độ lớn. Không có một tỉ lệ cố định nào cả.

## 9 · Thung lũng IsoFLOP: đo trực tiếp, không cần đường khớp

Cách thứ 2 của Chinchilla không cần dạng công thức nào: cố định $C$, huấn luyện vài cỡ mô hình, mỗi cỡ học đúng
$D = C/(6N)$ ký tự. Nhỏ quá thì thiếu sức chứa, lớn quá thì thiếu dữ liệu, nên mất mát theo $\ln N$ có dạng thung
lũng. Khớp một parabola theo $\ln N$ và đọc đáy của nó. Demo đã chạy 3 ngân sách như vậy, mỗi ngân sách 4–5 cỡ × 3 seed.
Ô dưới đọc đáy của từng thung lũng, và so tỉ lệ $D/N$ ở đó với tỉ lệ mà đường khớp ở mục 8 cho ở cùng ngân sách.

In [ ]:
def parabola(lnN, L):
    """L ≈ c₂(ln N)² + c₁ ln N + c₀; trả về N ở đáy và mất mát ở đáy (cách 2 của Chinchilla)."""
    c2, c1, c0 = np.polyfit(lnN, L, 2)
    return math.exp(-c1 / (2 * c2)), c0 - c1 * c1 / (4 * c2), (c2, c1, c0)


iso = {}
for C in BUDGETS:
    rs = [r for r in iso_runs if r["C"] == C]
    lnN = np.log([size[r["d"]]["total"] for r in rs])
    L = np.array([r["test"] for r in rs])
    N_par, L_par, coef = parabola(lnN, L)
    seeds = [C / 6 / parabola(lnN[s], L[s])[0] ** 2 for s in [np.array([r["seed"] == k for r in rs]) for k in range(3)]]
    iso[C] = {"runs": rs, "N": N_par, "L": L_par, "coef": coef, "seeds": seeds}
    N_fit, D_fit, _ = frontier(f0, C)
    print(
        f"C = {sci(C, 1)}: {len({r['d'] for r in rs})} cỡ × 3 seed; parabola: đáy N ≈ {vn(N_par, 0)}, "
        f"D/N ≈ {vn(C / 6 / N_par**2, 0)} (từng seed {vn(min(seeds), 0)}–{vn(max(seeds), 0)}); "
        f"đường khớp: N ≈ {vn(N_fit, 0)}, D/N ≈ {vn(D_fit / N_fit, 0)}"
    )
    assert abs(N_par / FITS["iso"][f"{C:g}"]["parabola"]["N"] - 1) < 1e-6, "khác với demo"
par_ratio = [C / 6 / iso[C]["N"] ** 2 for C in BUDGETS]
fit_ratio = [frontier(f0, C)[1] / frontier(f0, C)[0] for C in BUDGETS]
print(
    "D/N ở đáy, parabola:",
    " → ".join(vn(v, 0) for v in par_ratio),
    "· đường khớp:",
    " → ".join(vn(v, 0) for v in fit_ratio),
)
assert 80 < min(par_ratio) and max(par_ratio) < 120 and 135 < min(fit_ratio) and max(fit_ratio) < 215, "khác với demo"

Hình dưới vẽ ba thung lũng: mỗi ngân sách một màu và một hình điểm (trung bình 3 seed, thanh dọc là khoảng giữa
3 seed). Đường liền là parabola theo $\ln N$; đường gạch là thung lũng mà đường khớp của mục 5 dự đoán,
$L(N, C/(6N))$; vạch đứng đánh dấu đáy parabola.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.3))
for C, color, mk in zip(BUDGETS, (BLUE, ORANGE, GREEN), ("o", "s", "^")):
    v = iso[C]
    ds = sorted({r["d"] for r in v["runs"]})
    Ns = np.array([size[d]["total"] for d in ds])
    vals = [np.array([r["test"] for r in v["runs"] if r["d"] == d]) for d in ds]
    m = np.array([x.mean() for x in vals])
    yerr = [m - [x.min() for x in vals], [x.max() for x in vals] - m]
    ax.errorbar(Ns, m, yerr=yerr, fmt=mk, color=color, ms=7, mec="white", capsize=2, label=f"C = {sci(C, 1)} FLOP")
    xs = np.geomspace(Ns[0] / 1.5, Ns[-1] * 1.5, 100)
    c2, c1, c0 = v["coef"]
    ax.plot(xs, c2 * np.log(xs) ** 2 + c1 * np.log(xs) + c0, color=color, lw=1.4)
    ax.plot(xs, predict(f0, xs, C / (6 * xs)), color=color, lw=1.2, ls="--")
    ax.axvline(v["N"], color=color, lw=0.8, ls=":")
ax.set(xscale="log", xlabel="N (tham số, thang log)", ylabel="L thử (nat/ký tự)", ylim=(1.75, 2.6))
ax.set_title("Thung lũng IsoFLOP: liền = parabola theo ln N, gạch = đường khớp")
ax.legend(frameon=False)
ax.grid(alpha=0.3, which="both")
vn_axes(fig)
plt.show()

Thung lũng có thật: ở hai ngân sách lớn, một cỡ ở giữa thắng cả hai đầu, và đáy dịch sang phải khi $C$ tăng. Ở ngân
sách nhỏ nhất, d 16 và d 32 ngang nhau trong dao động giữa các seed, nên đáy nằm sát mép trái của các cỡ đã đo: đó là
lý do parabola từng seed ở đây trải từ 93 đến 166. Và hai cách đọc đáy **không thống nhất cả về chiều**:
- parabola cho tỉ lệ khoảng 115 → 93 → 85 ký tự mỗi tham số, tức *giảm* khi $C$ tăng;
- đường khớp cho 139 → 169 → 207, tức *tăng*.

Với 4–5 điểm mỗi thung lũng, đáy parabola rất nhạy với điểm ở hai đầu, còn đường khớp thì bị ràng buộc bởi dạng
công thức. Không cách nào “sai”: dữ liệu chỉ chưa đủ để chọn. Vì vậy demo nói “khoảng 85–115 ký tự mỗi tham số theo
các thung lũng IsoFLOP, và 140–210 theo đường khớp”, chứ không nói một con số.

Đây là **ký tự**, không phải token BPE. Một token GPT-2 dài trung bình vài ký tự (Bài 14), nên con số khoảng 100 này
không mâu thuẫn gì với “20 token mỗi tham số” của Chinchilla. Tỉ lệ tối ưu phụ thuộc dữ liệu, cách tách token, kiến
trúc, và cả cách ước lượng.

## 10 · Vì sao Kaplan khuyên tăng mô hình nhanh hơn?

Kaplan và cộng sự (2020) kết luận $N_\text{opt} \propto C^{0{,}73}$: phần lớn ngân sách nên dành cho mô hình. Chinchilla
tìm ra $a \approx 0{,}5$. Hai lời giải thích thường được nêu:
1. **Lịch learning rate** (Hoffmann và cộng sự): Kaplan đọc mất mát ở $D$ *giữa chừng* một lần chạy dài, có lịch
   learning rate dài hơn $D$, nên mất mát đó cao hơn so với một lần chạy có lịch kết thúc đúng ở $D$.
2. **Cách đếm $N$** (Pearce & Song, 2024): Kaplan không đếm tham số embedding. Ở mô hình nhỏ, embedding chiếm một phần
   lớn số tham số, nên bỏ nó đi làm cong số mũ.

Các lần chạy dài nhất của demo được đo cả ở giữa chừng ($2^{18} \dots 2^{21}$ ký tự), nên ta thử được cả hai ý trên
cùng các lần chạy.

In [ ]:
P_nonemb = points(grid, count="nonemb")
P_inter = points(grid, sched="inter")
diff = P_inter[:, 2] - P_all[:, 2]
part = P_all[:, 1] < DMAX
print(
    f"đọc giữa chừng cao hơn lịch vừa đúng D: trung bình {vn(diff[part].mean(), 3)}, "
    f"nhiều nhất {vn(diff[part].max(), 3)} nat ({part.sum()} lần chạy)"
)
fits_k = {"mọi tham số, lịch vừa đúng D": f0}
fits_k["không tính embedding (Kaplan)"] = fit(P_nonemb)
fits_k["đọc giữa chừng (Kaplan)"] = fit(P_inter)
for name, f in fits_k.items():
    a_k = f["beta"] / (f["alpha"] + f["beta"])
    print(f"{name:<30}: E {vn(f['E'], 2)} · α {vn(f['alpha'], 2)} · β {vn(f['beta'], 2)} → a = {vn(a_k, 2)}")
for key, f in (
    ("all-nonemb-sched", fits_k["không tính embedding (Kaplan)"]),
    ("all-total-inter", fits_k["đọc giữa chừng (Kaplan)"]),
):
    st = FITS["fits"][key]
    assert abs(f["alpha"] - st["alpha"]) < 2e-3 and abs(f["beta"] - st["beta"]) < 2e-3, "khác với demo"
share = {d: 1 - size[d]["nonemb"] / size[d]["total"] for d in (16, 128)}
print(f"embedding + lớp ra + vị trí: {pct(share[16], 0)} tham số của d 16, {pct(share[128], 0)} của d 128")

- **Không đếm embedding** đẩy $a$ từ 0,43 lên 0,50, tức về phía “tăng mô hình nhanh hơn”, cùng chiều với Kaplan.
  Lý do: bảng ký tự, vị trí và lớp ra chiếm 53% tham số của mô hình nhỏ nhất nhưng chỉ 7% của d 128, nên bỏ chúng
  đi làm các mô hình nhỏ “nhỏ hơn” nhiều hơn các mô hình lớn.
- **Đọc giữa chừng** cao hơn trung bình 0,047 nat, đúng như Hoffmann nói, nhưng ở đây nó đẩy $a$ theo chiều *ngược
  lại* (0,30), và kéo $E$ xuống 1,44. Lịch learning rate không giải thích được khác biệt Kaplan–Chinchilla trong quét
  này. Porian và cộng sự (2024) cũng thấy lịch giảm learning rate cẩn thận “not essential”. Theo họ, khác biệt đến từ
  ba điều khác: FLOP của lớp ra bị bỏ qua, warmup quá dài với mô hình nhỏ, và không chỉnh bộ tối ưu theo quy mô.

Bài học chung: số mũ của một quy luật mở rộng nhạy với những lựa chọn trông như chi tiết kỹ thuật (đếm gì, đọc mất
mát lúc nào, chỉnh $\eta$ ra sao).

## 11 · “20 token mỗi tham số” từ đâu ra?

Chinchilla (70 tỉ tham số, 1,4 nghìn tỉ token) học đúng 20 token mỗi tham số, và con số 20 thành quy tắc ngón tay cái.
Bài báo có ba cách ước lượng. Cách 1 và 2 cho $a \approx 0{,}5$, còn cách 3 là đường khớp như ta vừa làm, và in các
hằng số $E = 1{,}69$, $A = 406{,}4$, $B = 410{,}7$, $\alpha = 0{,}34$, $\beta = 0{,}28$. Mã TeX của bài có các giá trị chính
xác hơn. Besiroglu và cộng sự (2024) dựng lại dữ liệu từ hình vẽ trong bài và khớp lại.

### 🤔 Dự đoán trước
Với ngân sách của Gopher, $C = 5{,}76 \cdot 10^{23}$ FLOP, chính các hằng số in trong bài cho tỉ lệ tối ưu bao nhiêu
token mỗi tham số: khoảng 20, khoảng 50, hay khoảng 90?

In [ ]:
CH_FITS = {  # Hoffmann và cộng sự 2022, phụ lục D.2; mã TeX của bài; Besiroglu và cộng sự 2024, phương trình 4
    "in trong bài": {"E": 1.69, "A": 406.4, "B": 410.7, "alpha": 0.34, "beta": 0.28},
    "mã TeX": {"E": 1.6934, "A": 406.4, "B": 410.7, "alpha": 0.3392, "beta": 0.2849},
    "Besiroglu 2024": {"E": 1.8172, "A": 482.01, "B": 2085.43, "alpha": 0.3478, "beta": 0.3658},
}
assert [CH_FITS[k] for k in CH_FITS] == [CHIN["fits"][k] for k in ("paper", "tex", "refit")]  # cùng hằng số với demo
C_GOPHER = 5.76e23
ratio_gopher = {}
for name, f in CH_FITS.items():
    N, D, L = frontier(f, C_GOPHER)
    Nb, _ = brute(f, C_GOPHER, N)
    assert abs(math.log(Nb / N)) < 1e-3
    ratio_gopher[name] = D / N
    other = " · ".join(f"{sci(C, 0)}: {vn(frontier(f, C)[1] / frontier(f, C)[0], 0)}" for C in (1e21, 1e25, 1e27))
    print(
        f"{name:<15}: α = {vn(f['alpha'], 4)}, β = {vn(f['beta'], 4)}, a = {vn(f['beta'] / (f['alpha'] + f['beta']), 3)}; ở C của Gopher N_opt = {vn(N / 1e9, 1)} tỉ, "
        f"D_opt = {vn(D / 1e12, 2)} nghìn tỉ, D/N = {vn(D / N, 1)}  (ở {other})"
    )
assert abs(ratio_gopher["in trong bài"] - 92.6) < 0.1 and abs(ratio_gopher["Besiroglu 2024"] - 18.4) < 0.1, (
    "khác với demo"
)

Chính các hằng số in trong bài cho khoảng **93** token mỗi tham số ở ngân sách của Gopher, gần gấp năm lần con số 20
mà Chinchilla được huấn luyện. Chỉ dùng các số mũ chính xác hơn trong mã TeX ($\alpha$ 0,3392 thay vì 0,34, $\beta$
0,2849 thay vì 0,28) đã kéo tỉ lệ xuống 59: tỉ lệ nhạy với chữ số thứ hai, thứ ba của số mũ. Và vì $\alpha \ne \beta$,
tỉ lệ còn đổi theo $C$.

Bản khớp lại của Besiroglu và cộng sự cho khoảng **18**, và đổi chậm theo $C$ (22 ở $10^{21}$, 15 ở $10^{27}$) vì
$\alpha \approx \beta$. Họ chỉ ra
rằng bản gốc lấy *trung bình* Huber thay vì *tổng*, nên bộ tối ưu dừng quá sớm (mục 14 thử lại điều này). Họ cũng cho
rằng khoảng tin cậy trong bài hẹp đến mức khó tin. “20 token mỗi tham số” khớp với cách 1, cách 2 và bản khớp lại,
nhưng nó là quy tắc ngón tay cái, không phải hằng số tự nhiên.

Hình dưới vẽ tỉ lệ tối ưu theo ngân sách cho ba bộ hằng số (ba kiểu nét), cùng các mô hình thật đặt ở $C = 6ND$ của
chính chúng (hình thoi cam, có tên).

In [ ]:
models = {m["name"]: m for m in CHIN["models"]}
Cs = np.geomspace(1e19, 1e27, 60)
fig, ax = plt.subplots(figsize=(8.5, 4.4))
for (name, f), ls in zip(CH_FITS.items(), ("-", "--", ":")):
    ax.plot(Cs, [frontier(f, C)[1] / frontier(f, C)[0] for C in Cs], color=BLUE, ls=ls, lw=2, label=name)
OFFSET = {"GPT-3": (-36, 6), "Gopher": (8, -4), "Llama 3 8B": (8, -4), "Llama 3.1 405B": (-40, -14)}
for name, m in models.items():
    ax.plot(6 * m["N"] * m["D"], m["D"] / m["N"], "D", color=ORANGE, ms=7, mec="white")
    xy = (6 * m["N"] * m["D"], m["D"] / m["N"])
    ax.annotate(name, xy, xytext=OFFSET.get(name, (6, 4)), textcoords="offset points", fontsize=9)
ax.axhline(20, color=GREY, lw=1, ls="-.")
ax.text(2e26, 11, "20 token/tham số", color=GREY, fontsize=9, ha="center")
ax.set(
    xscale="log",
    yscale="log",
    xlabel="C (FLOP huấn luyện, thang log)",
    ylabel="D/N (token mỗi tham số)",
    ylim=(0.5, 4000),
)
ax.set_title("Tỉ lệ tối ưu tính toán theo ba bộ hằng số của Chinchilla")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0, 0.9))
ax.grid(alpha=0.3, which="both")
vn_axes(fig)
plt.show()
for name in ("GPT-3", "Gopher", "Chinchilla", "Llama 3 8B", "Llama 3 70B", "Llama 3.1 405B"):
    m = models[name]
    print(
        f"{name:<15}: N = {vn(m['N'] / 1e9, 1)} tỉ, D = {vn(m['D'] / 1e12, 2)} nghìn tỉ, D/N = {vn(m['D'] / m['N'], 1)}"
    )

Có ba điều đáng để ý trên hình (đường liền là tỉ lệ tối ưu theo từng bộ hằng số, hình thoi cam là mô hình thật):
- Gopher và GPT-3 nằm rất thấp, cỡ 1–2 token mỗi tham số: quá lớn so với dữ liệu, ở phía “thiếu dữ liệu” của thung
  lũng.
- Chinchilla nằm đúng trên đường của bản khớp lại.
- Các Llama 3 nằm *cao hơn* mọi đường, Llama 3 8B tới gần 1.900 token mỗi tham số. Đó không phải là sai lầm: mục
  kế tiếp tính vì sao.

Llama 3 cũng tự khớp quy luật trên dữ liệu của mình và chọn 402 tỉ tham số cho 16,55 nghìn tỉ token ở
$3{,}8 \cdot 10^{25}$ FLOP. Ô dưới tính tỉ lệ đó.

In [ ]:
l3 = CHIN["llama3Fit"]
print(
    f"Llama 3, đường khớp riêng: {vn(l3['N'] / 1e9, 0)} tỉ tham số, {vn(l3['D'] / 1e12, 2)} nghìn tỉ token ở "
    f"{sci(l3['C'], 1)} FLOP → D/N ≈ {vn(l3['D'] / l3['N'], 0)}; 6ND = {sci(6 * l3['N'] * l3['D'], 1)}"
)
N_r, D_r, _ = frontier(CH_FITS["Besiroglu 2024"], l3["C"])
print(f"bản khớp lại của Chinchilla ở cùng ngân sách: {vn(N_r / 1e9, 0)} tỉ tham số, D/N ≈ {vn(D_r / N_r, 0)}")

Cùng một dạng công thức, khác dữ liệu và bộ tách từ: 41 thay vì 17. (Cặp số ấy cho $6ND = 4 \cdot 10^{25}$, hơi lớn
hơn ngân sách $3{,}8 \cdot 10^{25}$ mà bài báo ghi, nên phép đếm FLOP của họ không đúng bằng $6ND$.) Tỉ lệ tối ưu là
một tính chất của *dữ liệu, cách tách token và kiến trúc*, không phải của “mô hình ngôn ngữ” nói chung.

## 12 · Tối ưu để huấn luyện chưa chắc tối ưu để dùng

Đường tối ưu tính toán trả lời câu hỏi “rẻ nhất để **huấn luyện**”. Nhưng một mô hình được dùng để sinh văn bản, có
khi hàng nghìn tỉ token, và mỗi token sinh ra tốn khoảng $2N$ FLOP (chỉ lượt xuôi, bỏ qua attention và KV cache,
Bài 17). Với $T$ token sẽ sinh, tổng chi phí là

$$\text{chi phí}(N) = 6 N D(N) + 2 N T,$$

với $D(N)$ là số token mà mô hình $N$ tham số cần để đạt một mức mất mát mục tiêu (giải từ đường khớp:
$D = \big(B / (L - E - A/N^\alpha)\big)^{1/\beta}$). Sardana và cộng sự (2024) đề xuất cực tiểu hóa đúng tổng này.

### 🤔 Dự đoán trước
Theo bản khớp lại của Chinchilla, mức mất mát của Llama 3 8B (15 nghìn tỉ token) có thể đạt bằng một mô hình tối ưu
tính toán lớn hơn, học ít token hơn. Mô hình đó lớn khoảng bao nhiêu, và phải sinh bao nhiêu token thì mô hình 8B mới
“hòa vốn”?

In [ ]:
def tokens_for(f, target, N):
    """Số token mà mô hình N tham số cần để đạt mất mát `target` (vô cùng nếu đáy E + A/N^α đã cao hơn)."""
    gap = target - f["E"] - f["A"] / N ** f["alpha"]
    return (f["B"] / gap) ** (1 / f["beta"]) if gap > 0 else math.inf


def golden(fn, lo, hi, iters=200):
    """Tìm cực tiểu của một hàm một đỉnh trên [lo, hi] bằng lát cắt vàng."""
    g = (math.sqrt(5) - 1) / 2
    a, b = lo, hi
    c, d = b - g * (b - a), a + g * (b - a)
    fc, fd = fn(c), fn(d)
    for _ in range(iters):
        if fc < fd:
            b, d, fd = d, c, fc
            c = b - g * (b - a)
            fc = fn(c)
        else:
            a, c, fc = c, d, fd
            d = a + g * (b - a)
            fd = fn(d)
    return (a + b) / 2


def cheapest(f, target, T):
    """N, D rẻ nhất (huấn luyện + sinh T token) để đạt mất mát target; tìm trên ln N."""
    lo = math.log((f["A"] / (target - f["E"])) ** (1 / f["alpha"])) + 1e-9  # N nhỏ nhất còn đạt được target

    def cost(lN):
        return 6 * math.exp(lN) * tokens_for(f, target, math.exp(lN)) + 2 * math.exp(lN) * T

    N = math.exp(golden(cost, lo, lo + 16))
    return N, tokens_for(f, target, N), cost(math.log(N))


fr = CH_FITS["Besiroglu 2024"]
m8 = models["Llama 3 8B"]
target = float(predict(fr, m8["N"], m8["D"]))
C8 = 6 * m8["N"] * m8["D"]
N0, D0, C0 = cheapest(fr, target, 0.0)
lNs = np.linspace(math.log(N0) - 3, math.log(N0) + 3, 600001)  # kiểm tra bằng cách dò
gap = target - fr["E"] - fr["A"] / np.exp(lNs) ** fr["alpha"]
ok = gap > 0  # N quá nhỏ thì đáy E + A/N^α đã cao hơn mục tiêu
grid_cost = np.full_like(lNs, np.inf)
grid_cost[ok] = 6 * np.exp(lNs[ok]) * (fr["B"] / gap[ok]) ** (1 / fr["beta"])
assert abs(lNs[grid_cost.argmin()] - math.log(N0)) < 1e-4 and grid_cost.min() >= C0 * (1 - 1e-12)
print(
    f"Llama 3 8B: {vn(m8['N'], 0)} tham số × {vn(m8['D'] / 1e12, 0)} nghìn tỉ token, "
    f"D/N = {vn(m8['D'] / m8['N'], 0)} (gần {vn(round(m8['D'] / m8['N'], -2), 0)})"
)
print(f"mức mất mát của nó (theo bản khớp lại): {vn(target, 3)} nat/token")
print(
    f"rẻ nhất để huấn luyện tới mức đó: {vn(N0 / 1e9, 1)} tỉ tham số × {vn(D0 / 1e9, 0)} tỉ token (D/N = {vn(D0 / N0, 0)})"
)
print(f"  huấn luyện rẻ hơn {vn(C8 / C0, 1)} lần; mỗi token sinh ra đắt hơn {vn(N0 / m8['N'], 1)} lần")
T_even = (C8 - C0) / (2 * (N0 - m8["N"]))
print(f"8B hòa vốn sau T = {sci(T_even, 2)} token sinh ra")
for T in (1e12, 1e13, 1e14):
    N, D, cost = cheapest(fr, target, T)
    print(f"  nếu sẽ sinh {sci(T, 0)} token: rẻ nhất là {vn(N / 1e9, 1)} tỉ tham số, D/N = {vn(D / N, 0)}")
assert abs(C8 / C0 - 5.50) < 0.01 and abs(N0 / m8["N"] - 4.22) < 0.01 and 1.1e13 < T_even < 1.2e13, "khác với demo"

Theo bản khớp lại, mức mất mát của Llama 3 8B đạt được rẻ nhất bằng một mô hình khoảng 34 tỉ tham số học khoảng 650
tỉ token, đúng 19 token mỗi tham số. Huấn luyện mô hình đó rẻ hơn 5,5 lần, nhưng mỗi token nó sinh ra đắt hơn
4,2 lần. Sau khoảng $1{,}1 \cdot 10^{13}$ token sinh ra, mô hình 8B “học quá” đã rẻ hơn tính cả hai phần. Càng định
sinh nhiều, mô hình rẻ nhất càng nhỏ và học càng lâu: với $10^{14}$ token, tỉ lệ tối ưu là 1.136 token mỗi tham số.

Lưu ý: hằng số của Chinchilla đo trên MassiveText với bộ tách từ của Gopher, không phải dữ liệu của Llama 3, nên các
con số trên chỉ để minh họa cách tính. Sardana và cộng sự (2024) đã huấn luyện thật tới $10^4$ token mỗi tham số và
thấy chất lượng vẫn tăng. MoE (Bài 24) là một cách khác để có nhiều tham số mà mỗi token vẫn rẻ.

## 13 · Năng lực “bộc phát”: do mô hình hay do thước đo?

Wei và cộng sự (2022) gọi một năng lực là “bộc phát” (emergent) nếu nó gần như không có ở mô hình nhỏ rồi xuất hiện
đột ngột ở mô hình lớn. Schaeffer và cộng sự (2023) chỉ ra rằng phần lớn những năng lực đó được đo bằng thước
**tất-cả-hoặc-không** như khớp chính xác cả chuỗi.

Demo đo cả hai loại thước trên các mô hình của ta ($D = 2^{22}$, 3 seed mỗi cỡ, 6 cỡ kể cả cỡ giữ riêng). Thước trơn
là bit mỗi ký tự. Thước tất-cả-hoặc-không là: sau một dấu cách, mô hình có đoán đúng (argmax) **mọi** ký tự của $k$
âm tiết tiếp theo không, tức giải mã tham lam có chép lại được đúng chúng không. Thước từng phần là tỉ lệ ký tự đúng
trong chính các đoạn ấy.

### 🤔 Dự đoán trước
Mô hình lớn nhất đoán đúng khoảng 53% số ký tự trong các đoạn 4 âm tiết (dài trung bình 18,5 ký tự). Nếu mỗi ký tự
đúng độc lập với xác suất 0,53, cả đoạn đúng với xác suất bao nhiêu? Trong 324.666 đoạn (gộp 18 mô hình), bạn đoán
có bao nhiêu đoạn khớp chính xác?

In [ ]:
EM_D = [16, 32, 48, 64, 96, 128]
em = {d: [r["em"] for r in grid + held if r["d"] == d and r["D"] == DMAX] for d in EM_D}
assert all(len(v) == 3 for v in em.values())


def avg(d, k):
    return float(np.mean([e[k] for e in em[d]]))


for d in EM_D:
    print(
        f"d {d:>3} ({vn(size[d]['total'], 0):>7} tham số): {vn(avg(d, 'bpc'), 3)} bit/ký tự · ký tự đúng {pct(avg(d, 'acc'), 1)} · "
        f"khớp 1 / 2 âm tiết {pct(avg(d, 'em1'), 2)} / {pct(avg(d, 'em2'), 3)}, "
        f"4 âm tiết {sum(round(e['em4'] * e['n4']) for e in em[d])} đoạn · "
        f"ký tự đúng trong đoạn 4 âm tiết {pct(avg(d, 'pc4'), 1)}"
    )
n4, len4 = em[16][0]["n4"], em[16][0]["len4"]
hits4 = sum(round(e["em4"] * e["n4"]) for v in em.values() for e in v)
n_models = sum(len(v) for v in em.values())
p_ind = avg(128, "pc4") ** len4
print(f"đoạn 4 âm tiết: {vn(n4, 0)} đoạn mỗi mô hình, dài trung bình {vn(len4, 1)} ký tự")
print(f"khớp chính xác: {hits4} lần trong {vn(n_models * n4, 0)} đoạn ({n_models} mô hình × {vn(n4, 0)})")
print(
    f"nếu từng ký tự độc lập: {vn(avg(128, 'pc4'), 3)}^{vn(len4, 1)} = {sci(p_ind, 1)}, tức {vn(p_ind * n4, 2)} trên {vn(n4, 0)} đoạn"
)
who = [f"d {r['d']} seed {r['seed']}" for r in grid + held if r["D"] == DMAX and r["em"]["em4"] > 0]
print("mô hình có đoạn 4 âm tiết khớp chính xác:", ", ".join(who))
assert hits4 == 2 and n_models * n4 == 324666, "khác với demo"

Hình dưới đặt hai thước cạnh nhau theo $N$ (thang log). Trái: bit mỗi ký tự, thước trơn. Phải: khớp chính xác 1, 2 và
4 âm tiết (ba màu, ba hình điểm), thước tất-cả-hoặc-không, cùng tỉ lệ ký tự đúng trong các đoạn 4 âm tiết (đường gạch,
trục phải).

In [ ]:
Ns_em = [size[d]["total"] for d in EM_D]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
a1.plot(Ns_em, [avg(d, "bpc") for d in EM_D], "o-", color=BLUE, ms=6)
a1.set(xscale="log", xlabel="N (tham số, thang log)", ylabel="bit/ký tự", title="Thước trơn: giảm đều")
for k, color, mk in ((1, BLUE, "o"), (2, ORANGE, "s"), (4, GREEN, "^")):
    a2.plot(
        Ns_em, [100 * avg(d, f"em{k}") for d in EM_D], marker=mk, color=color, ms=6, label=f"khớp chính xác {k} âm tiết"
    )
a2.set(xscale="log", xlabel="N (tham số, thang log)", ylabel="khớp chính xác (%)", title="Thước tất-cả-hoặc-không")
a3 = a2.twinx()
a3.plot(Ns_em, [100 * avg(d, "pc4") for d in EM_D], "--", color=GREY, lw=1.5, label="ký tự đúng trong 4 âm tiết (phải)")
a3.set_ylabel("ký tự đúng (%)", color=GREY)
a3.spines["top"].set_visible(False)
h1, l1 = a2.get_legend_handles_labels()
h2, l2 = a3.get_legend_handles_labels()
a2.legend(h1 + h2, l1 + l2, frameon=False, fontsize=8, loc="lower right", bbox_to_anchor=(1, 0.06))
for ax in (a1, a2):
    ax.grid(alpha=0.3, which="both")
plt.tight_layout()
vn_axes(fig)
plt.show()
steps = [avg(a, "bpc") - avg(b, "bpc") for a, b in itertools.pairwise(EM_D)]
print(
    f"N ×{vn(NH / size[16]['total'], 0)}: bit/ký tự {vn(avg(16, 'bpc'), 2)} → {vn(avg(128, 'bpc'), 2)}; "
    "mỗi bước lên cỡ kế tiếp bớt " + " · ".join(vn(x, 3) for x in steps)
)
assert all(x > 0 for x in steps) and all(a > b for a, b in itertools.pairwise(steps))  # giảm đều, không bước nhảy
print(
    f"ký tự đúng ×{vn(avg(128, 'acc') / avg(16, 'acc'), 2)}, nhưng khớp chính xác 1 âm tiết ×{vn(avg(128, 'em1') / avg(16, 'em1'), 1)}"
)

Cùng các mô hình, hai câu chuyện khác nhau:
- **Thước trơn** (bit mỗi ký tự) giảm từ 2,95 xuống 2,54 khi $N$ tăng 61 lần, không có bước nhảy nào; mỗi bước lên
  cỡ kế tiếp bớt ít hơn bước trước.
- **Thước tất-cả-hoặc-không** phóng đại cùng tiến bộ đó: tỉ lệ ký tự đúng chỉ tăng 1,14 lần, nhưng khớp chính xác
  1 âm tiết tăng 5,6 lần. Với 4 âm tiết, khớp chính xác gần như không bao giờ xảy ra: 2 lần trong 324.666 đoạn, cả
  hai ở một mô hình d 32, không phải mô hình lớn nhất. Ước lượng độc lập cho mô hình lớn nhất là 0,16 đoạn trên
  18.037, đúng cỡ đó.

Nếu tiếp tục tăng quy mô, tỉ lệ ký tự đúng nhích dần lên, và lũy thừa bậc 18,5 của nó sẽ vọt từ “gần 0” lên “đáng
kể” trong vài cỡ mô hình. Nhìn qua thước khớp chính xác, đó là một năng lực “bộc phát”, dù mô hình bên dưới tiến bộ
đều. Schaeffer và cộng sự (2023) thấy phần lớn (hơn 92%) các năng lực “bộc phát” trên BIG-Bench được đo bằng chấm
trắc nghiệm hoặc khớp chính xác chuỗi; đổi sang thước đo liên tục thì đường cong thường trơn.

Điều đó không có nghĩa mọi năng lực đều đoán trước được, và dự đoán **mất mát** cũng không tự cho ra dự đoán **độ chính
xác**. Llama 3 phải thêm một bước: từ FLOP ra log-likelihood trên bài kiểm tra, rồi từ log-likelihood ra độ chính
xác qua một đường sigmoid.

## 14 · Thử thay đổi

Các ô dưới chạy lại cùng hàm `fit()` với lựa chọn khác. Bạn có thể sửa giá trị rồi chạy lại.

### Ngưỡng Huber $\delta$
Chinchilla dùng $\delta = 10^{-3}$, nhỏ đến mức gần như mọi điểm nằm ở phần tuyến tính. Thử lớn hơn:

In [ ]:
for delta in (1e-3, 1e-2, 1e-1):
    f = fit(P_all, delta=delta)
    N12, D12, _ = frontier(f, 1e12)
    print(
        f"δ = {vn(delta, 3)}: E {vn(f['E'], 3)} · α {vn(f['alpha'], 3)} · β {vn(f['beta'], 3)}; "
        f"d 128 ở 2²²: {vn(predict(f, NH, DMAX), 3)}; D/N tối ưu ở 10¹²: {vn(D12 / N12, 0)}"
    )

Dự đoán cho mô hình giữ riêng gần như không đổi (lệch dưới 0,01 nat), nhưng tỉ lệ tối ưu nhảy từ 165 lên 230 khi
$\delta = 0{,}1$. Một lần nữa, đại lượng ta *đo được* bền vững, còn đại lượng ta *ngoại suy* thì không.

### Tổng hay trung bình?
Besiroglu và cộng sự (2024) cho rằng bản gốc lấy **trung bình** Huber thay vì **tổng**. Về toán, hai mục tiêu có cùng
cực tiểu. Nhưng trung bình nhỏ hơn 75 lần, gradient cũng vậy, nên một bộ tối ưu dừng khi gradient đủ nhỏ sẽ dừng
sớm hơn. Ô dưới cho BFGS dừng khi $|g| < 10^{-5}$ (thay vì $10^{-13}$) với cả hai mục tiêu.

In [ ]:
res_ms = {}
for name, scale in (("tổng", 1.0), ("trung bình", 1 / len(P_all))):
    f = fit(P_all, scale=scale, gtol=1e-5)
    res_ms[name] = f
    print(
        f"{name:<10}: α {vn(f['alpha'], 3)} · β {vn(f['beta'], 3)} · E {vn(f['E'], 3)}; tổng Huber {vn(f['obj'] * 1e4, 3)}·10⁻⁴"
    )
assert res_ms["trung bình"]["obj"] > 1.1 * res_ms["tổng"]["obj"]  # trung bình dừng ở một điểm tệ hơn hẳn

Lấy tổng, BFGS vẫn về đúng cực tiểu của mục 5. Lấy trung bình, nó dừng ở một điểm có mục tiêu cao hơn hẳn và số mũ
khác xa. Cùng dữ liệu, cùng công thức, chỉ khác một hằng số nhân trong hàm mục tiêu. Đây là một trong những lý do
Besiroglu và cộng sự đưa ra cho việc các hằng số in trong bài lệch khỏi cách 1 và 2.

### Lấy mẫu lại các lần chạy (bootstrap)
Các hằng số chắc chắn đến đâu? Ta rút có hoàn lại 75 lần chạy từ 75 lần chạy, khớp lại, và lặp vài lần. Demo đã làm
200 lần như vậy; ở đây ta làm 12 lần cho nhanh.

In [ ]:
boot_rng = np.random.default_rng(5)
boot = []
for _ in range(12):
    f = fit(P_all[boot_rng.integers(0, len(P_all), len(P_all))])
    N12, D12, _ = frontier(f, 1e12)
    boot.append((f["alpha"], f["beta"], f["beta"] / (f["alpha"] + f["beta"]), D12 / N12))
boot = np.array(boot)
for j, name in enumerate(("α", "β", "a = β/(α+β)", "D/N tối ưu ở 10¹²")):
    dg = 0 if j == 3 else 2
    print(f"{name:<18}: {vn(boot[:, j].min(), dg)}–{vn(boot[:, j].max(), dg)} qua 12 lần lấy mẫu lại")
A_CI_CHINCHILLA = (0.454, 0.455)  # Hoffmann và cộng sự 2022, bảng 2: khoảng 10–90% của a, cách 3
print(
    f"Chinchilla, cách 3: a từ {vn(A_CI_CHINCHILLA[0], 3)} đến {vn(A_CI_CHINCHILLA[1], 3)}; "
    f"khoảng của ta rộng gấp ~{vn(round(np.ptp(boot[:, 2]) / np.ptp(A_CI_CHINCHILLA), -1), 0)} lần"
)
b200 = FITS["boot"]["total"]
print(
    f"demo, 200 lần (10–90%): α {vn(b200['alpha'][0], 2)}–{vn(b200['alpha'][2], 2)} · "
    f"D/N ở 10¹² {vn(b200['ratio']['1e+12'][0], 0)}–{vn(b200['ratio']['1e+12'][2], 0)}"
)

Chỉ cần đổi *những lần chạy nào* được dùng, $\alpha$ đã dao động cỡ ±0,1, và tỉ lệ tối ưu ở $C = 10^{12}$ có thể là
130 hay 210 ký tự mỗi tham số. Khoảng 10–90% mà Hoffmann và cộng sự báo cho $a$ ở cách 3 hẹp hơn khoảng 100 lần.
Besiroglu và cộng sự cho rằng muốn hẹp như vậy cần hơn $6 \cdot 10^5$ lần chạy, trong khi Chinchilla huấn luyện hơn
400 mô hình.

### Chỉ một seed
Mỗi ô của lưới có 3 seed. Thử khớp chỉ seed 0 (25 lần chạy) và so với 3 seed:

In [ ]:
P_seed0 = points([r for r in grid if r["seed"] == 0])
f_s0 = fit(P_seed0)
print(
    f"seed 0: E {vn(f_s0['E'], 3)} · α {vn(f_s0['alpha'], 3)} · β {vn(f_s0['beta'], 3)}; d 128 ở 2²²: {vn(predict(f_s0, NH, DMAX), 3)}"
)
print(
    f"3 seed: E {vn(f0['E'], 3)} · α {vn(f0['alpha'], 3)} · β {vn(f0['beta'], 3)}; d 128 ở 2²²: {vn(predict(f0, NH, DMAX), 3)}"
)
assert abs(f_s0["alpha"] - FITS["fits"]["seed0-total-sched"]["alpha"]) < 2e-3, "khác với demo"

Một seed (25 lần chạy) cho số mũ khác 3 seed một chút, nhưng dự đoán cho mô hình giữ riêng chỉ lệch 0,001 nat.

## 15 · Giới hạn, và bước tiếp theo

Những điều notebook này *không* nói được:
- **Vùng đo rất nhỏ.** Mô hình của ta có 13.952–850.880 tham số. Mô hình nhỏ nhất của Chinchilla, 70 triệu, đã lớn
  gấp 82 lần mô hình lớn nhất ở đây. Mọi đường khớp chỉ đáng tin trong vùng đã đo, và ngoại suy càng xa càng
  chịu ảnh hưởng của những chữ số số mũ mà ta vừa thấy là không chắc.
- **Dữ liệu là vô hạn.** Mọi lần chạy đều thấy mỗi ký tự đúng một lần. Khi dữ liệu có hạn và phải học lặp nhiều
  lượt, giá trị của mỗi token lặp lại giảm dần (Muennighoff và cộng sự, 2023), và công thức cần thêm một số hạng.
- **Mất mát không phải năng lực.** Như mục 13, đi từ mất mát đến độ chính xác trên một nhiệm vụ cần thêm một bước.
- **Một loại kiến trúc.** Hằng số đổi theo kiến trúc. Bài 24 thay đổi chính phép tính $C \approx 6ND$: với
  Mixture of Experts, mỗi token chỉ đi qua vài “chuyên gia”, nên số tham số và số phép tính mỗi token tách rời nhau.

In [ ]:
rng_ch = CHIN[
    "range"
]  # Hoffmann và cộng sự 2022: "70 million to over 16 billion parameters on 5 to 500 billion tokens"
print(f"ở đây: {vn(size[16]['total'], 0)}–{vn(NH, 0)} tham số, {vn(DS[0] / 1e6, 2)}–{vn(DMAX / 1e6, 2)} triệu ký tự")
print(
    f"Chinchilla: {vn(rng_ch['Nmin'] / 1e6, 0)} triệu – {vn(rng_ch['Nmax'] / 1e9, 0)} tỉ tham số; mô hình nhỏ nhất của họ "
    f"lớn gấp {vn(rng_ch['Nmin'] / NH, 0)} lần mô hình lớn nhất ở đây"
)

## Tóm tắt
- Huấn luyện tốn $C \approx 6ND$: mỗi trọng số làm 1 phép nhân–cộng ở lượt xuôi và 2 ở lượt ngược. GPT-2 small
  (124.439.808 tham số) đếm đủ ra 797.870.592 FLOP mỗi token, gấp 1,07 lần $6N$; attention chiếm 7%.
- Dạng của Chinchilla, $L = E + A/N^\alpha + B/D^\beta$, khớp bằng Huber trên $\ln L$, tóm 75 lần chạy bằng 5 hằng số
  (lệch 0,035 nat). Phần còn giảm được là luật lũy thừa; $L$ thì cong về đáy (1,83 với mô hình nhỏ nhất).
- Đường khớp đoán một mô hình lớn gấp 1,7 lần chưa từng thấy, sai 0,017 nat. Dự đoán bền hơn số mũ: bốn tập con cho
  $\beta$ từ 0,47 đến 0,77, nhưng dự đoán chỉ chênh 0,024.
- Tối ưu tính toán có dạng đóng $N_\text{opt} \propto C^{\beta/(\alpha+\beta)}$. Tỉ lệ tối ưu của ta: 85–115 ký tự mỗi
  tham số theo thung lũng IsoFLOP, 140–210 theo đường khớp, và hai cách còn ngược chiều nhau.
- “20 token mỗi tham số” không phải định luật: hằng số in trong bài Chinchilla cho 93, bản khớp lại cho 18, Llama 3
  cho 41.
- Llama 3 8B học gần 1.900 token mỗi tham số vì nó sẽ sinh rất nhiều: rẻ hơn mô hình tối ưu tính toán sau khoảng
  $10^{13}$ token.
- “Bộc phát” phần lớn là do thước đo: bit mỗi ký tự giảm đều, còn khớp chính xác 4 âm tiết là 2 lần trong 324.666.

## Câu hỏi ôn tập
1. Vì sao một trọng số tốn 6 FLOP cho mỗi token huấn luyện, và vì sao bảng embedding gần như không tốn FLOP nào?
2. Trên thang log–log, mất mát theo $D$ của mô hình nhỏ nhất cong chứ không thẳng. Vì sao? Đại lượng nào mới thẳng?
3. Với $C = 6ND$ cố định, chứng minh rằng nếu $\alpha = \beta$ thì tỉ lệ $D/N$ tối ưu không phụ thuộc $C$.
4. Hằng số in trong bài Chinchilla cho khoảng 93 token mỗi tham số, nhưng chính Chinchilla học 20. Nêu hai lý do con số
   20 vẫn được coi là hợp lý.
5. Tỉ lệ tối ưu của các mô hình ký tự ở đây là khoảng 100 ký tự mỗi tham số. Điều đó có mâu thuẫn với “20 token mỗi
   tham số” không?
6. Một nhóm định phục vụ $10^{14}$ token với mô hình của mình. Nên chọn mô hình nhỏ hơn hay lớn hơn mô hình tối ưu tính
   toán, và học nhiều hay ít token hơn?
7. Một năng lực đo bằng khớp chính xác “xuất hiện đột ngột” ở cỡ mô hình nào đó. Bạn sẽ kiểm tra gì trước khi gọi nó
   là bộc phát?

<details><summary>Gợi ý đáp án</summary>

1. Lượt xuôi: 1 MAC ($xW$); lượt ngược: 2 MAC (gradient theo đầu vào và theo trọng số). Mỗi MAC là 2 FLOP. Bảng
   embedding chỉ được tra (chọn một hàng), không nhân ma trận; trừ khi nó cũng là lớp ra như ở GPT-2.
2. Vì mất mát có đáy $E + A/N^\alpha$ mà thêm dữ liệu không vượt qua. Phần $L - E - A/N^\alpha = B/D^\beta$ mới thẳng,
   dốc $-\beta$.
3. $N_\text{opt} \propto C^{\beta/(\alpha+\beta)} = C^{1/2}$ và $D_\text{opt} \propto C^{1/2}$, nên $D/N = G^{-2}$, một
   hằng số.
4. Cách 1 và 2 của chính bài báo cho $a \approx 0{,}5$ và tỉ lệ cỡ 20; bản khớp lại (Besiroglu và cộng sự, 2024) cho
   18, và chỉ ra rằng bản gốc dừng tối ưu quá sớm. Đường IsoFLOP cũng phẳng gần đáy, nên lệch vài lần quanh tối ưu
   tốn ít.
5. Không: một bên là ký tự, một bên là token BPE dài vài ký tự; tỉ lệ còn phụ thuộc dữ liệu, kiến trúc và cách ước
   lượng (ở đây hai cách cho 85–115 và 140–210).
6. Nhỏ hơn và học nhiều token hơn: mỗi token sinh ra tốn khoảng $2N$ FLOP, nên với $T$ lớn, phần sinh lấn át phần
   huấn luyện (mục 12: với $10^{14}$ token, khoảng 8,8 tỉ tham số và 1.136 token mỗi tham số).
7. Đo cùng các mô hình bằng một thước liên tục (log-likelihood, tỉ lệ ký tự hoặc token đúng). Nếu nó tăng đều, bước
   nhảy đến từ thước tất-cả-hoặc-không chứ không phải từ mô hình.
</details>

## Tìm hiểu thêm
- J. Kaplan và cộng sự (2020), [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361).
- J. Hoffmann và cộng sự (2022), [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556)
  (Chinchilla): cách 1–3, phụ lục D.2 (đường khớp), bảng A4 ($6ND$).
- T. Besiroglu và cộng sự (2024), [Chinchilla Scaling: A replication attempt](https://arxiv.org/abs/2404.10102), và
  [bài blog của Epoch AI](https://epoch.ai/blog/chinchilla-scaling-a-replication-attempt).
- T. Pearce & J. Song (2024), [Reconciling Kaplan and Chinchilla Scaling Laws](https://arxiv.org/abs/2406.12907):
  cách đếm tham số embedding.
- T. Porian và cộng sự (2024), [Resolving Discrepancies in Compute-Optimal Scaling of Language Models](https://arxiv.org/abs/2406.19146).
- N. Sardana và cộng sự (2024), [Beyond Chinchilla-Optimal: Accounting for Inference](https://arxiv.org/abs/2401.00448).
- Llama Team (2024), [The Llama 3 Herd of Models](https://arxiv.org/abs/2407.21783), mục 3.2.1: quy luật mở rộng
  riêng và bước dự đoán độ chính xác.
- N. Muennighoff và cộng sự (2023), [Scaling Data-Constrained Language Models](https://arxiv.org/abs/2305.16264).
- J. Wei và cộng sự (2022), [Emergent Abilities of Large Language Models](https://arxiv.org/abs/2206.07682); R. Schaeffer,
  B. Miranda & S. Koyejo (2023), [Are Emergent Abilities of Large Language Models a Mirage?](https://arxiv.org/abs/2304.15004)
- Epoch AI, [Training tokens per parameter of open-weight models](https://epoch.ai/data-insights/training-tokens-per-parameter).
- Stanford CS336 (2025), [Lecture 9: Scaling laws basics](https://raw.githubusercontent.com/stanford-cs336/spring2025-lectures/main/nonexecutable/2025%20Lecture%209%20-%20Scaling%20laws%20basics.pdf);
  L. Weng (2026), [Scaling Laws, Carefully](https://lilianweng.github.io/posts/2026-06-24-scaling-laws/).

**Bài tiếp theo:** [Bài 24 · Mixture of Experts (MoE)](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mixture-of-experts/):
nhiều tham số mà mỗi token vẫn rẻ, khi $C \approx 6ND$ chỉ đếm những chuyên gia mà token thật sự đi qua.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")